# Exploring the Original Data
- Goal: understand the dataset before making big architectural changes relating to the project


## Loading the DataSet
- Inspected the first fow.
- Inspected the column names

In [2]:
from datasets import load_dataset
from collections import Counter

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("iberu/RunBugRun")

print(ds.keys())
print(ds['train'].column_names)
print(ds['train'][0])

C:\Users\Home\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['train', 'validation', 'test'])
['id', 'buggy_submission_id', 'fixed_submission_id', 'problem_id', 'user_id', 'buggy_code', 'fixed_code', 'labels', 'change_count', 'line_hunks', 'errors']
{'id': 6581, 'buggy_submission_id': 3585, 'fixed_submission_id': 3586, 'problem_id': 'p00000', 'user_id': 'u243053043', 'buggy_code': 'for i in range(1, 10):\n    for j in range(1, 10):\n        print("{} x {} = {}".format(i, j, i*j))', 'fixed_code': 'for i in range(1, 10):\n    for j in range(1, 10):\n        print("{}x{}={}".format(i, j, i*j))', 'labels': ['literal.string.change', 'call.arguments.change', 'io.output.change'], 'change_count': 1, 'line_hunks': 1, 'errors': None}


## Looking at the Top 20 Labels and their Frequency
- Found: There are many different labels with many examples. All examples come from a taxonomy type of label

In [ ]:
label_counter = Counter()

for row in ds['train']:
    if row['labels']:
        label_counter[row['labels'][0]] += 1

print('\nTop 20 Labels:')
for label, count in label_counter.most_common(20):
    print(label, count)

print('\n Number of Unique Primary Labels:', len(label_counter))


Top 20 Labels:
literal.number.integer.change 11636
call.add 10883
identifier.change 10300
assignment.value.change 10035
literal.string.change 9816
call.remove 9452
call.arguments.change 7901
expression.operator.compare.change 7812
control_flow.branch.if.condition.change 7122
expression.operation.binary.remove 5252
control_flow.loop.range.bounds.upper.change 4200
misc.opposites 4132
call.arguments.add 3560
expression.operator.arithmetic.change 3341
expression.operation.binary.add 2760
assignment.variable.change 2570
assignment.add 2083
assignment.change 1748
identifier.replace.add 1313
identifier.replace.remove 817

 Number of Unique Primary Labels: 59


## Investigating Label Counts
- Looked into missing labels
- Looked into the frequency of how many labels existed per example
- Found that there was a large enough dataset to support a single label predicting model

In [4]:
#investigating: how many labels per example
label_count_dist = Counter()
missing_labels = 0

for row in ds['train']:
    if row['labels'] is None:
        missing_labels += 1
        continue
    label_count_dist[len(row["labels"])] += 1

print("Labels per example:")
for num_labels, count in sorted(label_count_dist.items()):
    print(f'{num_labels} labels: {count}')

Labels per example:
0 labels: 6391
1 labels: 35926
2 labels: 35538
3 labels: 22481
4 labels: 18224
5 labels: 6982
6 labels: 2875
7 labels: 1044
8 labels: 345
9 labels: 104
10 labels: 30
11 labels: 8
12 labels: 3
13 labels: 1


## Frequency for Top Level Categories
- Found: a small subset of values and decided to remove those with less than 1000 examples (difficult to learn)
- Learned: there was a somewhat healthy distribution to work with

In [5]:
#investigating: how many top level categories exist
unique_label_category = Counter()

for row in ds['train']:
    if row['labels'] == None:
            continue
    for label in row['labels']:
        category = label.split('.')[0]
        unique_label_category[category] += 1

for name, count in sorted(unique_label_category.items()):
    print(f'{name} - {count}')

assignment - 29113
call - 85217
control_flow - 48811
expression - 53111
function - 1154
identifier - 26032
io - 20751
literal - 34933
misc - 6384
try_catch - 6
type_conversion - 248
variable_access - 5182


## Verifying all Examples were Python
- This test was conducted because this came from a dataset that originally had other languages involved and featured labels like try_catch (python is try-except)
- Chose random examples and looked at the code
- Found: all random examples were in Python, so I concluded that most of the code must be in Python

In [ ]:
for example in [1, 20, 50, 100, 200, 4000, 20000]:
     code = ds['train'][example]['buggy_code']
     print(f'{example} - Code: {code}')

1 - Code: for x in range(1,10):
    for y in range(1,10):
        print("%d x %d = %d" % (x, y, x*y))
20 - Code: for a in range(1,9):
	for b in range(1,9):
		print(a,"x",b,"=",a*b,sep="")

50 - Code: for i in range(1, 10, 1):
    for j in range(1, 10, 1):
        print(i, "*", j, "=", i * j, sep='')
100 - Code: #!/usr/bin/env python3
#coding: utf-8

# Volume0 - 0001 (http://judge.u-aizu.ac.jp/onlinejudge/description.jsp?id=0001)

li = []
for x in range(10):
    s = input()
    int(s)
    li.append(s)

li.sort()
li.reverse()

for y in range(3):
    print(li[y])
200 - Code: deta_set_count = int(input())
for _ in range(deta_set_count):
    k = list(map(int, input().split()))
    k= k.sorted()
    if k[0]**2 + k[1]**2 == k[2]**2:
        print('YES')
    else:
        print('NO')
4000 - Code: i = 0
while True :
  x = int(input())
  if(x == 0) :
     break
  else :
    i += 1
    print("Case ", i, ":", x, sep = "")
    

20000 - Code: 
n = int(input())
s = str(input())

if n >= len(s):
    

## Try-Catch Label
- Looked at try_catch labels more in-depth to confirm that they were in Python
- Found: all examples were in Python

In [ ]:
for row in ds['train']:
    if row['labels'] is None:
        continue
    for label in row['labels']:
        if label.split('.')[0] == 'try_catch':
            print(f"{label} - Code: {row['buggy_code']}")

try_catch.catch.exception.change - Code: import math

def main():
    data = []
    while 1:
        try:
            n = input().split()
            a = int(n[0])
            b = int(n[1])
            ans = int(math.log10(a+b)+1)
            data.append(ans)
        except IndexError:
            #EOF
            break
            
    for i in data:
        print(i)

if __name__ == "__main__":
    main()
try_catch.catch.exception.change - Code: b = ["the", "this", "that"]
while True:
        try:
                a = input()
        except IndexError:
                break
        for y in range(26):
                for x in b:
                        if x in a:
                                print(a)
                                break
                a = list(a)
                print(a)
                for x in range(len(a)):
                        if a[x] == "z":
                                a[x] = "a"
                        elif "a" <= a[x] < "z":
                         

## Counting the Frequency of Unique Labels per Example
- Found: most examples had 1 unique idenifier, further supporting the idea that the model could be trained using only single label examples to predict one label

In [8]:
#investigating: how many unique labels per example
unique_amount_label_counter = Counter()

for row in ds['train']:
    if row['labels'] is None:
        continue
    else:
        unique_label_in_row = set()
        for label in row['labels']:
            label = label.split('.')[0]
            unique_label_in_row.add(label)
            
        unique_amount_label_counter[len(unique_label_in_row)] += 1

for key, value in unique_amount_label_counter.most_common(5):
    print(f'Unique Identifier Amount: {key} - Frequency: {value}')

Unique Identifier Amount: 1 - Frequency: 47378
Unique Identifier Amount: 2 - Frequency: 29432
Unique Identifier Amount: 3 - Frequency: 28923
Unique Identifier Amount: 4 - Frequency: 11945
Unique Identifier Amount: 0 - Frequency: 6391


## Viewing Row with No Labels
- Found: some of these have labels that are meaning no change
- Removed from the dataset because these rows do not have a label

In [15]:
rows_with_empty_label = []
for row in ds['train']:
    if not row['labels']:
        rows_with_empty_label.append((row['buggy_code'], row['fixed_code']))

print(f'This is the number of rows in the rows without any labels: {len(rows_with_empty_label)}')

for i in range(5): #the range can be changed as long as it is less than the len of the list
    print(f'Buggy Code {i}:\n {rows_with_empty_label[i][0]}\n Fixed Code {i}:\n {rows_with_empty_label[i][1]}')

This is the number of rows in the rows without any labels: 10144
Buggy Code 0:
 num = [int(input()) for i in range(10)]
num.sort(reverse=True)
for i in range[0:3]:
    print(num[i])

 Fixed Code 0:
 num = [int(input()) for i in range(10)]
num.sort(reverse=True)
for i in range(3):
    print(num[i])

Buggy Code 1:
 def quicksort(array):
    if len(array) < 2:
        return array
    else:
        pivot = array[0]
        less = [i for i in array[1:] if i <= pivot]

        greater = [i for i in array[1:] if i > pivot]

        return quicksort(less) + [pivot] + quicksort(greater)


array = []
for _ in range(10):
    array.append(int(input('input integer')))

sorted_array = quicksort(array)
for i in range(3):
    print(sorted_array[9-i])
 Fixed Code 1:
 def quicksort(array):
    if len(array) < 2:
        return array
    else:
        pivot = array[0]
        less = [i for i in array[1:] if i <= pivot]

        greater = [i for i in array[1:] if i > pivot]

        return quicksort(less